## Import stuff


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.AudioProcessing import (
    AudioProcessingError,
    load_audio,
    normalize_audio,
    convert_to_mono,
    resample_audio,
    validate_audio,
    remove_silence,
    prepare_audio,
)

## Audio Data
Loading and details

In [ ]:
audio_path = Path("../data/raw/example.wav")
assert audio_path.exists(), f"File not found: {audio_path}"
print(audio_path)

original_sample_rate, original_audio = load_audio(audio_path)

print("Sample rate:", original_sample_rate)
print("Shape:", original_audio.shape)
print("Dtype:", original_audio.dtype)
print("Minimum:", original_audio.min())
print("Maximum:", original_audio.max())

## Change to Mono
change audio channel to mono
use Unstandardize Audio data

In [ ]:
mono_audio = convert_to_mono(original_audio)

print("Original shape:", original_audio.shape)
print("Mono shape:", mono_audio.shape)
assert mono_audio.shape[0] == original_audio.shape[0]
assert mono_audio.ndim == 1

## Audio Normalize
Audio is not a floating point
use Unstandardize Audio data

In [ ]:
normalized_audio = normalize_audio(mono_audio)

print("Dtype:", normalized_audio.dtype)
print("Minimum:", normalized_audio.min())
print("Maximum:", normalized_audio.max())

assert np.max(np.abs(normalized_audio)) <= 1.0
assert normalized_audio.dtype == np.float64
assert np.all(np.isfinite(normalized_audio))

# Sampling Change
Sampling rate is different, change to standardize sampling rate
use Unstandardize Audio data

In [ ]:
resampled_audio, resampled_rate = resample_audio(
    normalized_audio,
    original_sample_rate=original_sample_rate,
    target_sample_rate=16000,
)

print("Original rate:", original_sample_rate)
print("New rate:", resampled_rate)
print("Original samples:", len(normalized_audio))
print("New samples:", len(resampled_audio))

assert resampled_rate == 16000
assert resampled_audio.ndim == 1
assert np.all(np.isfinite(resampled_audio))

original_duration = len(normalized_audio) / original_sample_rate
new_duration = len(resampled_audio) / resampled_rate

print("Original duration:", original_duration)
print("New duration:", new_duration)

assert np.isclose(original_duration, new_duration, atol=0.01)

## Use silence remove function
use Unstandardize Audio data
use standardize audio data

In [ ]:
trimmed_audio = remove_silence(
    resampled_audio,
    resampled_rate,
    threshold=0.01,
)

print("Before:", len(resampled_audio))
print("After:", len(trimmed_audio))

validate_audio(
    trimmed_audio,
    resampled_rate,
    expected_sample_rate=16000,
)

Plotting

In [ ]:
time_before = np.arange(len(resampled_audio)) / resampled_rate
time_after = np.arange(len(trimmed_audio)) / resampled_rate

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(time_before, resampled_audio)
axes[0].set_title("Before silence removal")

axes[1].plot(time_after, trimmed_audio)
axes[1].set_title("After silence removal")

plt.tight_layout()
plt.show()

## validation

In [ ]:
validate_audio(
    resampled_audio,
    resampled_rate,
    expected_sample_rate=16000,
    minimum_duration_seconds=0.1,
)

print("Audio passed validation.")

In [ ]:
try:
    validate_audio(
        np.array([]),
        16000,
    )
except AudioProcessingError as error:
    print("Correctly rejected:", error)

## testing complete pipeline

In [ ]:
prepared_audio, prepared_rate = prepare_audio(
    audio_path,
    expected_sample_rate=16000,
    apply_silence_removal=False,
)

print("Prepared shape:", prepared_audio.shape)
print("Prepared rate:", prepared_rate)
print("Prepared dtype:", prepared_audio.dtype)
print("Prepared duration:", len(prepared_audio) / prepared_rate)

validate_audio(
    prepared_audio,
    prepared_rate,
    expected_sample_rate=16000,
)

In [ ]:
prepared_audio_trimmed, prepared_rate_trimmed = prepare_audio(
    audio_path,
    expected_sample_rate=16000,
    apply_silence_removal=True,
)

## Syntethic Testing

In [ ]:
sample_rate = 16000

leading_silence = np.zeros(4000)
speech = np.full(8000, 0.2)
trailing_silence = np.zeros(4000)

synthetic_audio = np.concatenate(
    (leading_silence, speech, trailing_silence)
)

trimmed = remove_silence(
    synthetic_audio,
    sample_rate,
    threshold=0.01,
)

print("Original duration:", len(synthetic_audio) / sample_rate)
print("Trimmed duration:", len(trimmed) / sample_rate)